# 10 逆解析：目標物性から配合候補を探索する

このNotebookでは、目標物性を満たす配合候補を二段階で探します。

1. 過去に実測された配合から、目標に最も近いものを探す
2. 同じ主PVC・主可塑剤系列の過去配合を補間し、順解析モデルで仮想候補を採点する

仮想候補は実験前の探索案です。原料の使用可否、配合合計、加工性、コスト、法規はこのNotebookでは保証しません。実験へ進める前に必ず配合担当者が処方を確認してください。

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(project_root / 'src'))

import pandas as pd
from IPython.display import display

from plastech.inverse_design import (
    DEFAULT_INVERSE_TARGETS,
    build_candidate_library,
    candidate_summary,
    fit_forward_models,
    generate_interpolated_candidates,
    rank_historical_candidates,
    specifications_frame,
)
from plastech.modeling import column_metadata


## 目標物性

これは製品仕様が未確定な段階の初期設定です。硬度・100%モジュラス・比重は目標値±許容幅、脆化温度は **-33℃以下** を要求します。製品用途が決まったら、このセルで値を変更します。

In [2]:
specifications = DEFAULT_INVERSE_TARGETS
display(specifications_frame(specifications))


,column,property,objective,target,tolerance,weight,requirement
0,S03010_030010,硬度 HDA (0s),target,78.00,3.00,1.0,78 ± 3
1,S06010_060030,100%モジュラス (JIS2 200mm/min),target,7.50,1.50,1.0,7.5 ± 1.5
2,S13010_130010,脆化温度 (JIS K6723),upper_bound,-33.00,3.00,1.0,<= -33
3,S01010_010010,比重 (23℃),target,1.34,0.05,0.7,1.34 ± 0.05


## 配合から物性を予測するモデルを学習する

逆向きに直接回帰するのではなく、物性ごとの順解析モデルを作ります。後の候補は、このモデルが予測した物性と目標の差で採点します。

In [3]:
forward_models = fit_forward_models(
    specifications,
    dataset_name='processed_main',
    model_name='random_forest',
    n_estimators=400,
    random_state=42,
)

pd.DataFrame(
    {
        'property': [spec.name for spec in specifications],
        'measured_rows_used_for_training': [
            len(forward_models.prepared[spec.column].y) for spec in specifications
        ],
        'feature_columns': [
            len(forward_models.prepared[spec.column].feature_columns) for spec in specifications
        ],
    }
)


FileNotFoundError: config/data_catalog.local.json がありません。config/data_catalog.example.json を参考にローカル用の対応表を作成してください。

## 実測済みの過去配合を探す

4物性をすべて実測している配合だけを候補ライブラリにします。ここでは実測値で順位付けするため、最初の候補確認として最も確実です。`predicted_*` 列は、同じ配合を順解析モデルがどの程度再現するかの確認用です。

In [ ]:
library = build_candidate_library(
    forward_models,
    dataset_name='processed_main',
    require_all_observed=True,
)
historical_ranking = rank_historical_candidates(library, forward_models)

print(f'全目標物性を実測済みの配合数: {len(library):,}')
print(f'実測値が全目標を満たす配合数: {historical_ranking["observed_meets_all"].sum():,}')
display(candidate_summary(historical_ranking, forward_models, top_n=20))


全目標物性を実測済みの配合数: 2,483
実測値が全目標を満たす配合数: 80


,candidate_type,row_id,primary_pvc,primary_plasticizer,observed_score,observed_meets_all,predicted_score,predicted_meets_all,observed__S03010_030010,predicted__S03010_030010,observed__S06010_060030,predicted__S06010_060030,observed__S13010_130010,predicted__S13010_130010,observed__S01010_010010,predicted__S01010_010010
0,historical,CP0000000002417,PVC1000 / TH1000(GW) (PVC.15),フタル酸エステル系 / DOP (可塑剤オイル類.29),0.068108,True,0.174255,True,78.0,79.125000,7.5,7.599250,-35.0,-34.002500,1.322,1.325459
1,historical,CP0000000000502,PVC1300 / TH1300(JG) (PVC.23),フタル酸エステル系 / DINP (可塑剤オイル類.30),0.131712,True,0.146104,True,78.0,77.927500,7.0,6.916000,-40.0,-39.840000,1.329,1.330923
2,historical,CP1000000009335,PVC1300 / TH1300(JG) (PVC.23),フタル酸エステル系 / DINP (可塑剤オイル類.30),0.131712,True,0.146104,True,78.0,77.927500,7.0,6.916000,-40.0,-39.840000,1.329,1.330923
3,historical,CP0000000000958,PVC1300 / TH1300(JG) (PVC.23),トリメリット酸系 / TOTM (可塑剤オイル類.24),0.205586,True,0.257283,True,77.0,76.169500,7.3,7.339500,-36.0,-34.895000,1.319,1.323230
4,historical,CP0000000002767,PVC2251- / TH2500(SG) (PVC.32),フタル酸エステル系 / DINP (可塑剤オイル類.30),0.207568,True,0.278916,True,78.0,77.117500,6.6,6.400250,-38.0,-36.938750,1.328,1.340332
5,historical,CP0000000000041,PVC1300 / TH1300(JG) (PVC.23),フタル酸エステル系 / DINP (可塑剤オイル類.30),0.216757,True,0.334191,False,78.0,78.604167,7.2,7.009017,-34.0,-34.170875,1.383,1.390557
6,historical,CP1000000008263,PVC1000 / TH1000(GW) (PVC.15),フタル酸エステル系 / DINP (可塑剤オイル類.30),0.224505,True,0.199070,True,77.0,77.887500,7.3,6.894616,-36.0,-37.685000,1.314,1.318895
7,historical,CP0000000002534,PVC1300 / TH1300(JG) (PVC.23),フタル酸エステル系 / DOP (可塑剤オイル類.29),0.227568,True,0.409024,False,78.0,76.714125,6.3,5.933161,-33.0,-34.427500,1.343,1.342872
8,historical,CP0000000002690,PVC2000 / TH2000(SW) (PVC.30),トリメリット酸系 / TOTM (可塑剤オイル類.24),0.228468,True,0.230964,True,79.0,78.410875,6.9,6.653750,-37.0,-37.767500,1.332,1.329040
9,historical,CP0000000000501,PVC1300 / TH1300(JG) (PVC.23),フタル酸エステル系 / DINP (可塑剤オイル類.30),0.236036,True,0.263837,True,78.0,77.457500,6.4,6.446500,-39.0,-38.452500,1.350,1.346645


## 仮想配合を生成して採点する

実測ランキング上位20件を起点に、同一の主PVC・主可塑剤を持つ配合と 25% / 50% / 75% で補間します。`interpolated_virtual` は未実測の候補です。`predicted_meets_all` は順解析予測上で仕様を満たすことを意味し、実測合格を意味しません。

In [ ]:
virtual_candidates = generate_interpolated_candidates(
    library,
    historical_ranking,
    forward_models,
    seed_count=20,
    partners_per_seed=3,
    fractions=(0.25, 0.5, 0.75),
    random_state=42,
)

print(f'生成した仮想候補数: {len(virtual_candidates):,}')
print(f'予測上で全目標を満たす候補数: {virtual_candidates["predicted_meets_all"].sum():,}')
display(candidate_summary(virtual_candidates, forward_models, top_n=20))


生成した仮想候補数: 180
予測上で全目標を満たす候補数: 43


,candidate_type,row_id,parent_a,parent_b,parent_a_fraction,primary_pvc,primary_plasticizer,predicted_score,predicted_meets_all,predicted__S03010_030010,predicted__S06010_060030,predicted__S13010_130010,predicted__S01010_010010
0,interpolated_virtual,virtual::CP0000000001093::CP0000000000532::0.75,CP0000000001093,CP0000000000532,0.75,PVC1300 / TH1300(JG) (PVC.23),フタル酸エステル系 / DINP (可塑剤オイル類.30),0.068188,True,78.567500,7.447958,-38.835417,1.337969
1,interpolated_virtual,virtual::CP0000000002690::CP0000000002240::0.25,CP0000000002690,CP0000000002240,0.25,PVC2000 / TH2000(SW) (PVC.30),トリメリット酸系 / TOTM (可塑剤オイル類.24),0.110075,True,77.481667,7.510500,-37.570000,1.323750
2,interpolated_virtual,virtual::CP1000000009335::CP1000000007634::0.75,CP1000000009335,CP1000000007634,0.75,PVC1300 / TH1300(JG) (PVC.23),フタル酸エステル系 / DINP (可塑剤オイル類.30),0.121527,True,77.930708,6.936400,-38.451000,1.343630
3,interpolated_virtual,virtual::CP1000000009335::CP1000000009866::0.75,CP1000000009335,CP1000000009866,0.75,PVC1300 / TH1300(JG) (PVC.23),フタル酸エステル系 / DINP (可塑剤オイル類.30),0.143517,True,78.237000,6.877000,-39.217500,1.342620
4,interpolated_virtual,virtual::CP0000000000502::CP0000000000501::0.75,CP0000000000502,CP0000000000501,0.75,PVC1300 / TH1300(JG) (PVC.23),フタル酸エステル系 / DINP (可塑剤オイル類.30),0.145359,True,77.855000,6.893750,-39.555000,1.333905
5,interpolated_virtual,virtual::CP0000000000502::CP1000000008490::0.75,CP0000000000502,CP1000000008490,0.75,PVC1300 / TH1300(JG) (PVC.23),フタル酸エステル系 / DINP (可塑剤オイル類.30),0.192058,True,79.212833,7.483625,-37.843500,1.361101
6,interpolated_virtual,virtual::CP0000000002690::CP0000000002240::0.50,CP0000000002690,CP0000000002240,0.50,PVC2000 / TH2000(SW) (PVC.30),トリメリット酸系 / TOTM (可塑剤オイル類.24),0.197969,True,78.566351,6.689750,-37.740000,1.340253
7,interpolated_virtual,virtual::CP1000000008263::CP1000000001883::0.75,CP1000000008263,CP1000000001883,0.75,PVC1000 / TH1000(GW) (PVC.15),フタル酸エステル系 / DINP (可塑剤オイル類.30),0.201712,True,78.265000,7.314833,-40.281667,1.301818
8,interpolated_virtual,virtual::CP0000000000502::CP0000000000501::0.50,CP0000000000502,CP0000000000501,0.50,PVC1300 / TH1300(JG) (PVC.23),フタル酸エステル系 / DINP (可塑剤オイル類.30),0.210203,True,77.885000,6.518500,-38.904375,1.333923
9,interpolated_virtual,virtual::CP0000000002690::CP0000000002240::0.75,CP0000000002690,CP0000000002240,0.75,PVC2000 / TH2000(SW) (PVC.30),トリメリット酸系 / TOTM (可塑剤オイル類.24),0.211054,True,77.943375,6.552250,-37.505000,1.330701


## 上位仮想候補の配合を確認する

上位候補と、その2つの親配合の配合量を比較します。ここで非ゼロ原料の増減を確認し、配合担当者が採用可否を判断します。

In [ ]:
selected = virtual_candidates.iloc[0]
parent_a = library.set_index('row_id').loc[selected['parent_a']]
parent_b = library.set_index('row_id').loc[selected['parent_b']]

feature_columns = sorted(
    {column for prepared in forward_models.prepared.values() for column in prepared.feature_columns}
)
metadata = column_metadata('processed_main').set_index('column')
recipe_review = pd.DataFrame(
    {
        'column': feature_columns,
        'display_name': [metadata.at[c, 'display_name'] for c in feature_columns],
        'parent_a': parent_a[feature_columns].astype(float).values,
        'parent_b': parent_b[feature_columns].astype(float).values,
        'virtual_candidate': selected[feature_columns].astype(float).values,
    }
)
recipe_review = recipe_review[recipe_review[['parent_a', 'parent_b', 'virtual_candidate']].ne(0).any(axis=1)]
recipe_review = recipe_review.sort_values('virtual_candidate', ascending=False)

print('選択候補:', selected['row_id'])
print('親配合A:', selected['parent_a'])
print('親配合B:', selected['parent_b'])
display(recipe_review)


選択候補: virtual::CP0000000001093::CP0000000000532::0.75
親配合A: CP0000000001093
親配合B: CP0000000000532


,column,display_name,parent_a,parent_b,virtual_candidate
11,PVC.23,PVC1300 / TH1300(JG) (PVC.23),100.00,100.0,100.0000
123,可塑剤オイル類.30,フタル酸エステル系 / DINP (可塑剤オイル類.30),61.00,75.0,64.5000
85,充填剤.2,未処理重質炭カル / NS400 (充填剤.2),20.00,55.0,28.7500
110,可塑剤オイル類.17,エポキシ化大豆油 / S300K (可塑剤オイル類.17),3.00,0.0,2.2500
154,安定剤金属石鹸.1,Ba系複合安定剤 / アデカAC190 (安定剤金属石鹸.1),1.50,0.0,1.1250
153,安定剤金属石鹸,Ba系複合安定剤 / アデカAP539 (安定剤金属石鹸),1.00,0.0,0.7500
374,樹脂改質剤加工助剤.67,アクリル系加工助剤 / メタブレンL1000 (樹脂改質剤加工助剤.67),0.00,2.5,0.6250
317,安定剤金属石鹸.86,無機酸鉛 / スタビFGT2 (安定剤金属石鹸.86),0.00,1.8,0.4500
194,安定剤金属石鹸.138,ベンゾトリアゾール系 / JF77 (安定剤金属石鹸.138),0.40,0.0,0.3000
429,滑剤.43,パラフィン炭化水素系 / ビスコワックス115 (滑剤.43),0.20,0.0,0.1500


## この結果の使い方

- `historical` 候補: 実測値のある既存処方。まず再現実験・用途適合の確認に使う。
- `interpolated_virtual` 候補: 予測上の探索案。実測、加工性、コスト、法規、原料調達を確認してから採用する。
- 予測値が目標を満たしても、モデルの誤差は残る。特に未知のPVC・可塑剤系列に広げる場合は、候補数を絞りすぎず実験結果を再学習へ戻す。
- 製品仕様が決まったら、先頭の `specifications` を更新し、同じ手順で再実行する。